Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Complex Analysis Lite

> ⚠️ **Draft — pending instructor review.** Visuals execute; proofs need a human pass before teaching. Remove this banner after review.

Two surgical sessions delivering exactly what the transform notebooks defer: what poles *are*, why the region of convergence matters, and how residues invert Laplace and $z$-transforms. Not a complex analysis course — a toolkit extraction.

## 1. Pre-requisites

[Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S3–S4 (you've met $H(s)$, $H(z)$); comfort with complex numbers as $re^{i\theta}$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

---
### 🕐 Session 1 of 2 — *Analyticity & Poles* (~35 min)
**Goal:** understand what makes complex differentiability special; read pole diagrams fluently.
**Builds on:** [DSP Foundations](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S3. &nbsp; **Feeds into:** Session 2 (residues).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Analyticity & Poles</b></summary>

**Timing (~35 min).** 10 min why complex differentiability is so strong · 8 min poles and the ROC · 12 min the landscape demo · 5 min buffer.

**Frame the workshop honestly at the start.** This is not a complex analysis course — it is a *toolkit extraction*. The transform notebooks kept deferring three things: what poles are, why the ROC matters, and how residues invert transforms. Two sessions, those three things, nothing else. Saying that up front prevents students from expecting (or fearing) a full treatment of contour integration.

**Board first — make complex differentiability feel extreme.** In real analysis the derivative is one limit, from two directions. In the complex plane $h \to 0$ can approach from *infinitely many* directions and the limit must agree for all of them. That is an enormously stronger demand, and the payoff is enormous too: analytic functions are automatically infinitely differentiable, equal to their Taylor series, and **rigid** — knowing the function on any small patch determines it everywhere. Ask the room whether any real function has that property; none does, and the contrast is what makes the theory worth having.

**Then connect it to why transforms behave.** $H(s)$ and $H(z)$ are analytic wherever their defining series converges, which is why manipulating them algebraically is safe. Where they *fail* to be analytic — the poles — is where all the system's personality lives. That reframing ("the interesting content is at the singularities") is the through-line of both sessions.

**Teach the landscape picture; it is the most valuable image here.** $|H(z)|$ over the complex plane is a terrain: poles are tent-poles pushing it up, zeros are pins holding it down. The frequency response is then **the altitude profile along a walk around the unit circle**. Once students hold that, they can read a pole-zero plot into a frequency response by eye: a pole near the circle at angle $\omega_0$ means the walk passes close to a tent-pole and the response spikes there; a zero *on* the circle means the walk steps on a pin and the response nulls.

**Ask the room.** "Where should I put a pole to build a resonator at $\omega_0$?" At radius just inside 1, angle $\omega_0$ — and closer to the circle means sharper resonance and longer ringing. That is [Filter Design](../../Intro_DSP/Filter_Design.ipynb)'s pole placement, and it is also the vocal-tract formant model from [Audio DSP](../../Intro_DSP/Audio_Speech_DSP.ipynb). One picture, several workshops.

**The ROC connects to something already proved.** The series $\sum h[n]z^{-n}$ converges on an annulus bounded by pole radii — which is [Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb)' radius of convergence, made two-sided. Students who have done that workshop should recognise the ROC as an old theorem rather than a new rule.

**Draft status.** This notebook still carries the ⚠️ review banner — the visuals execute, but the proofs want a human pass before the sessions are taught or recorded. Leave the banner until that review happens.
</details>

## 2. Analytic Functions

💡 **Intuition.** Complex differentiability is *outrageously* stronger than the real kind: the limit $\lim_{h\to 0} \frac{f(z+h) - f(z)}{h}$ must agree for $h$ approaching from **every direction** in the plane. Functions passing this test ('analytic') are automatically infinitely differentiable, equal to their Taylor series, and rigid — knowing one patch determines the whole. Transfer functions $H(s)$, $H(z)$ are analytic wherever their defining sum converges; the places they *fail* — **poles** — carry all the system's personality.

**Poles and the ROC.** A rational $H(z) = \frac{B(z)}{A(z)}$ blows up at the roots of $A$ (poles). The defining series $\sum_n h[n] z^{-n}$ converges on an annulus bounded by pole radii — the **region of convergence** ([Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb)' radius-of-convergence, now two-sided). The decisive readings:

- Causal & stable ⇔ all poles strictly inside the unit circle, ROC includes $|z| = 1$ — the criterion [Filter Design](../../Intro_DSP/Filter_Design.ipynb) checks with `zplane`.
- Pole near the circle at angle $\omega_0$ ⇒ resonance: $|H(e^{j\omega})|$ spikes as $e^{j\omega}$ passes close by.

In [2]:
# |H(z)| as a landscape: poles are tent-poles, zeros are pins; the unit circle walks the terrain
b = np.array([1.0, -1.0])                 # zero at z=1
a = np.array([1.0, -1.6, 0.72])           # poles at 0.8 e^{±j0.4π}... compute:
poles = np.roots(a); zeros = np.roots(b)

re, im = np.meshgrid(np.linspace(-1.4, 1.4, 400), np.linspace(-1.4, 1.4, 400))
z = re + 1j * im
Hmag = np.abs(np.polyval(b, z) / np.polyval(a, z))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
im0 = axes[0].imshow(np.log10(np.clip(Hmag, 1e-2, 1e2)), extent=[-1.4, 1.4, -1.4, 1.4], origin="lower", cmap="magma")
th = np.linspace(0, 2*np.pi, 200)
axes[0].plot(np.cos(th), np.sin(th), "c--", linewidth=1)
axes[0].plot(poles.real, poles.imag, "wx", markersize=9); axes[0].plot(zeros.real, zeros.imag, "wo", mfc="none")
axes[0].set_title("log|H(z)|: poles = tent-poles, circle = the walk")

w = np.linspace(0, np.pi, 500)
axes[1].plot(w / np.pi, np.abs(np.polyval(b, np.exp(1j*w)) / np.polyval(a, np.exp(1j*w))))
axes[1].axvline(np.angle(poles[0]) / np.pi, color="crimson", linestyle=":", label="pole angle")
axes[1].set_xlabel("ω/π"); axes[1].set_title("|H(e^{jω})|: the walk's altitude profile"); axes[1].legend()
plt.tight_layout(); plt.show()
print("poles at radius", np.abs(poles).round(3), "→ stable (inside unit circle)")

poles at radius [0.849 0.849] → stable (inside unit circle)


/tmp/ipykernel_2021715/713533306.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The left panel is $\log|H(z)|$ as a **landscape** over the complex plane: bright ridges rising at the poles, a dark point where the zero pins it to the floor, and a dashed unit circle drawn across the terrain. The right panel is the ordinary frequency response — and it is nothing more than **the altitude profile along that circular walk**.

That correspondence is the most useful picture in this workshop. Poles are tent-poles pushing the surface up; zeros are pins holding it down; the frequency response is what you experience walking the unit circle. Everything about reading pole-zero diagrams follows from it:

- A pole *near* the circle means the walk passes close to a tent-pole, so $|H(e^{j\omega})|$ spikes there — the red dotted line marks the pole angle, and the peak sits on it.
- The closer the pole to the circle, the sharper the peak and the longer the system rings.
- A zero *on* the circle means the walk steps directly on a pin, giving an exact null. That is how notch filters work, and here the zero at $z = 1$ is why the response vanishes at DC.

**The printed check.** Poles at radius **0.849** — inside the unit circle, so the system is stable. That is the entire stability criterion, and it is visible geometrically rather than algebraically: the tent-poles are inside the walk, so the walk never climbs to infinity.

**Why any of this is well-defined.** $H(z)$ is *analytic* everywhere except at its poles, and complex differentiability is an extraordinarily strong condition — the limit $\lim_{h\to0}(f(z+h)-f(z))/h$ must agree for $h$ approaching from every direction in the plane. Functions that pass are automatically infinitely differentiable, equal to their Taylor series, and rigid: knowing one patch determines the whole function. That rigidity is why we can manipulate transfer functions algebraically and trust the results, and why the *only* interesting places are the ones where analyticity fails.

**The ROC is an old theorem in new clothes.** The series $\sum_n h[n]z^{-n}$ converges on an annulus bounded by pole radii — precisely the radius-of-convergence result from [Sequences & Series](../Analysis/Numerical_Sequences_and_Series.ipynb), made two-sided because the sum runs over negative $n$ as well. Combined with the geometry above: causal *and* stable means all poles strictly inside the unit circle with the ROC containing $|z| = 1$, which is exactly what [Filter Design](../../Intro_DSP/Filter_Design.ipynb)'s `zplane` check verifies and what [Foundations 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s ROC session made a choice between.

---
### 🕐 Session 2 of 2 — *Residues & Inverse Transforms* (~40 min)
**Goal:** compute contour integrals by reading off residues; invert z-transforms by hand.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Residues & Inverse Transforms</b></summary>

**Timing (~40 min).** 10 min Cauchy and why loops give zero · 10 min residues · 10 min the worked inversion · 10 min the modes demo.

**Board first — the astonishing fact, stated before the machinery.** Integrate an analytic function around any closed loop and you get **exactly zero**. Not small, not approximately — zero, regardless of the loop's shape. That is Cauchy's theorem, and it is worth letting the room sit with it, because it means an analytic function has no "circulation" anywhere it is analytic. So the only way a contour integral can be nonzero is if the loop encircles a point where analyticity *fails*.

**Then the punchline.** All the value of a contour integral collapses to bookkeeping at the singularities, and each pole contributes through exactly **one number** — its residue. The integral is $2\pi i$ times the sum of enclosed residues. An infinite amount of functional detail reduces to a finite list of coefficients. That is why inverse transforms, which look like hard integrals, turn out to be partial fractions.

**Do the worked example slowly; it is the payoff.** $H(z) = z/(z-a)$ with $|z| > |a|$. Then $H(z)z^{n-1} = z^n/(z-a)$ has one simple pole at $z = a$, residue $a^n$, so $h[n] = a^n u[n]$. Point out what just happened: **the first line of every DSP transform table, derived rather than memorised.** Students who have been looking things up in tables for two workshops find it satisfying that the table is a consequence of one theorem.

**Emphasise where the ROC enters.** The contour must lie *in the ROC*, and that choice determines which poles are enclosed — which is exactly why the same algebraic $H(z)$ with different ROCs inverts to different signals, as [Foundations 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) showed. The ROC is not decoration; it selects the contour and therefore the answer.

**The second demo is the conceptual close of the workshop.** A pole pair at $re^{\pm j\theta}$ produces an impulse response $r^n \sin((n+1)\theta)/\sin\theta$ — and the agreement is *exactly* zero, not approximately. Make sure the room reads the two parameters physically: **$r$ is the decay rate, $\theta$ is the ringing frequency**. So a pole location is not an abstract root of a denominator; it is a decay-and-oscillation pair you can read off directly. Poles literally *are* the system's modes.

**Ask the room.** "What happens as $r \to 1$?" The decay slows and the ringing lasts forever; at $r = 1$ it never decays, and beyond it explodes. That is Session 1's stability criterion arriving from the time domain instead of the frequency domain — same boundary, different view. If [RNNs](../../Intro_Time_Series/Intro_RNN.ipynb) are on the syllabus, it is also the vanishing/exploding gradient condition.

**Note for Laplace.** The same machinery works along the Bromwich contour: close it leftward, sum residues, and you get sums of $e^{s_k t}$. Continuous-time modes, same theorem.
</details>

## 3. The Residue Theorem

💡 **Intuition.** Integrating an analytic function around a closed loop gives **zero** (Cauchy) — unless the loop encircles poles. Each pole contributes only through one number, its **residue** (the $\frac{1}{z - z_0}$ coefficient of the local expansion): the integral is $2\pi i$ × (sum of enclosed residues). All the detail of the function collapses to bookkeeping at its singularities — which is why inverse transforms, seemingly hard integrals, reduce to partial fractions.

**Statement.** For $f$ analytic inside and on a simple closed contour $\Gamma$ except at poles $z_k$ inside: $\oint_\Gamma f(z)\, dz = 2\pi i \sum_k \mathrm{Res}_{z_k} f$, with $\mathrm{Res}_{z_0} f = \lim_{z \to z_0} (z - z_0) f(z)$ for simple poles.

**Inverse z-transform.** $h[n] = \frac{1}{2\pi i} \oint H(z) z^{n-1} dz$ over a contour in the ROC — i.e. *sum the residues of $H(z) z^{n-1}$ at the enclosed poles*.

**Worked example.** $H(z) = \frac{z}{z - a}$, $|z| > |a|$ (causal). For $n \ge 0$: $H(z) z^{n-1} = \frac{z^n}{z-a}$ has one simple pole at $z = a$ with residue $a^n$. Hence $h[n] = a^n u[n]$ — the geometric/exponential pair every DSP table opens with, now *derived* rather than memorized. (Same machinery inverts Laplace along the Bromwich contour: close it leftward, sum residues, get sums of $e^{s_k t}$ — poles literally *are* the system's modes.)

In [3]:
# Verify the residue inversion numerically: contour-integrate H(z)z^{n-1} on |z|=1
a = 0.7
M = 4096
th = 2 * np.pi * np.arange(M) / M
z = np.exp(1j * th)                                  # unit circle (in the ROC since |a|<1)
H = z / (z - a)

h = [np.mean(H * z**n) .real for n in range(12)]     # (1/2πi)∮ H z^{n-1} dz = mean over circle of H·zⁿ
print("contour integral h[n]:", np.round(h, 5))
print("residue prediction aⁿ:", np.round(a ** np.arange(12), 5))

contour integral h[n]: [1.      0.7     0.49    0.343   0.2401  0.16807 0.11765 0.08235 0.05765
 0.04035 0.02825 0.01977]
residue prediction aⁿ: [1.      0.7     0.49    0.343   0.2401  0.16807 0.11765 0.08235 0.05765
 0.04035 0.02825 0.01977]


**What just happened.** Two rows of numbers, identical to five decimal places. The top row was obtained by **numerically integrating** $H(z)z^{n-1}$ around the unit circle — 4096 points, no theory. The bottom row is $a^n$, obtained by **reading one residue off a pole**. A hard-looking contour integral and a one-line algebraic answer agree exactly.

**Why the residue theorem collapses the work so drastically.** Cauchy's theorem says a closed-loop integral of an analytic function is *exactly zero*, whatever the loop's shape — an analytic function has no circulation anywhere it is analytic. So the only source of a nonzero integral is a point where analyticity fails, and each such pole contributes through exactly **one number**, its residue. The integral is $2\pi i$ times the sum of enclosed residues. An infinite amount of functional detail collapses to a short list of coefficients.

That is why inverse transforms, which are defined as contour integrals, turn out in practice to be partial fractions. The formidable-looking definition
$$h[n] = \frac{1}{2\pi i}\oint H(z)z^{n-1}\,dz$$
is, operationally, "list the enclosed poles and add up their residues."

**And the worked case is the first line of every DSP table.** $H(z) = z/(z-a)$ gives $H(z)z^{n-1} = z^n/(z-a)$, one simple pole at $z = a$, residue $\lim_{z\to a}(z-a)\cdot z^n/(z-a) = a^n$. So $h[n] = a^n u[n]$ — the geometric/exponential pair, **derived rather than looked up**. Every entry in those tables is a residue computation somebody did once.

**Note where the ROC does its work.** The contour had to lie inside the ROC, which here ($|a| = 0.7 < 1$) permits the unit circle. That choice determines *which poles are enclosed*, and therefore the answer. This is precisely why the same algebraic $H(z)$ with a different ROC inverts to a different signal, as [Foundations 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s causal-versus-anticausal demo showed. The ROC is not bookkeeping attached to the transform — it selects the contour, and the contour selects the answer.

One implementation note worth reading: `np.mean(H * z**n)` is the contour integral in disguise. Parameterising $z = e^{j\theta}$ turns $\oint H(z)z^{n-1}dz$ into an average of $H\cdot z^n$ over the circle, since $dz = jz\,d\theta$ and the $j$ and $2\pi$ cancel against the $1/2\pi i$ prefactor. The numerical method and the theory are computing the same object by different routes, which is what makes the agreement a genuine check.

In [4]:
# And the modes story: poles of a damped resonator ARE its impulse response ingredients
from scipy import signal as sig
b2, a2 = [1.0, 0.0, 0.0], [1.0, -1.4, 0.85]         # H(z)=z²/(z²−1.4z+0.85): poles r e^{±jθ}
# (scipy tf coefficients are in DESCENDING powers of z — b=[1] alone would mean z⁻²·H, a 2-sample delay)
poles2 = np.roots(a2); r, theta = np.abs(poles2[0]), np.angle(poles2[0])
n_ax = np.arange(60)
_, h_true = sig.dimpulse((b2, a2, 1), n=60)
h_true = np.squeeze(h_true)
mode = (r ** n_ax) * np.sin(theta * (n_ax + 1)) / np.sin(theta)   # residue formula for this pair

plt.figure(figsize=(8, 2.6))
plt.stem(n_ax, h_true, basefmt=" ", label="impulse response")
plt.plot(n_ax, mode, "r-", alpha=0.7, label="pole-mode formula  rⁿ·sin((n+1)θ)/sinθ")
plt.legend(); plt.title(f"poles at r={r:.2f}, θ={theta:.2f} rad — decay rate and ring frequency, read off the pole")
plt.tight_layout(); plt.show()
print("max |difference|:", np.abs(h_true - mode).max().round(10))

max |difference|: 0.0


/tmp/ipykernel_2021715/4120182771.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** `max |difference|: 0.0` — **exactly** zero, not small. The closed-form expression $r^n \sin((n+1)\theta)/\sin\theta$, derived from nothing but the two pole locations, reproduces the system's impulse response bit-for-bit across all 60 samples.

That exactness matters. This is not a fit or an approximation that happens to be good; the residue calculation *is* the impulse response, and `dimpulse` is computing the same numbers by running the difference equation forward. Two entirely different routes — recursion in the time domain, residues in the complex plane — to identical floating-point values.

**Read the formula physically, because this is the conceptual close of the workshop.** The pole pair sits at $re^{\pm j\theta}$, and the impulse response is $r^n \sin((n+1)\theta)/\sin\theta$:

- **$r$ is the decay rate.** The envelope is $r^n$, so the response dies with a time constant set by how far the pole sits from the origin.
- **$\theta$ is the ringing frequency.** The oscillation is $\sin((n+1)\theta)$, so the pole's *angle* is literally the frequency at which the system rings.

So a pole location is not an abstract root of a denominator polynomial. It is a **decay-and-oscillation pair, readable directly**: radius tells you how long it rings, angle tells you at what pitch. Poles *are* the system's modes, and an impulse response is the sum of its poles' modes with residues as coefficients.

**Which closes the loop with Session 1.** There, poles were tent-poles in a landscape and their angle set where the frequency response peaked. Here, the same angle sets the ringing frequency of the impulse response. Those are the same fact seen from the two sides of the transform — a resonance in frequency *is* a slowly-decaying oscillation in time.

**And stability appears again, now from the time domain.** As $r \to 1$ the envelope $r^n$ decays ever more slowly; at $r = 1$ it never decays; beyond it, $r^n$ grows without bound. That is exactly Session 1's "poles inside the unit circle" criterion, arrived at by watching the impulse response instead of the pole diagram. It is also the same geometric-series boundary that governs [RNN](../../Intro_Time_Series/Intro_RNN.ipynb) gradients and [Filter Design](../../Intro_DSP/Filter_Design.ipynb)'s stability check — one inequality, appearing wherever something is raised to the $n$th power.

**A note on the code comment.** The remark that scipy's coefficients are in *descending* powers of $z$ is not pedantry: writing `b2 = [1.0]` instead of `[1.0, 0.0, 0.0]` would silently give $z^{-2}H(z)$, a two-sample delay, and the mode formula would then appear to disagree. Coefficient-ordering conventions are a genuine source of confusion across libraries, and this is a good place to notice it.

## 4. Conclusion

Analyticity makes transforms rigid enough to trust; poles are where the personality lives; residues turn inverse transforms into partial-fraction bookkeeping; and a system's impulse response is literally the sum of its poles' modes.

---
## Where next

- [Foundations of Signal Processing 2](../../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the z-transform track, now with its inversion machinery.
- [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — pole placement as sculpture.